In [3]:
import pandahouse as ph

Задание 2. SQL
2.1 Очень усердные ученики.

2.1.1 Условие

Образовательные курсы состоят из различных уроков, каждый из которых состоит из нескольких маленьких заданий. Каждое такое маленькое задание называется "горошиной".

Назовём очень усердным учеником того пользователя, который хотя бы раз за текущий месяц правильно решил 20 горошин.

2.1.2 Задача

Дана таблица default.peas:

Название атрибута	Тип атрибута	Смысловое значение
st_id	int	ID ученика
timest	timestamp	Время решения карточки
correct	bool	Правильно ли решена горошина?
subject	text	Дисциплина, в которой находится горошина


Необходимо написать оптимальный запрос, который даст информацию о количестве очень усердных студентов.NB! Под усердным студентом мы понимаем студента, который правильно решил 20 задач за текущий месяц.

2.2 Оптимизация воронки

2.2.1 Условие

Образовательная платформа предлагает пройти студентам курсы по модели trial: студент может решить бесплатно лишь 30 горошин в день. Для неограниченного количества заданий в определенной дисциплине студенту необходимо приобрести полный доступ. Команда провела эксперимент, где был протестирован новый экран оплаты.

2.2.2 Задача

Дана таблицы: default.peas (см. выше), default.studs:

Название атрибута	Тип атрибута	Смысловое значение
st_id	int	 ID ученика
test_grp	text	 Метка ученика в данном эксперименте
и default.final_project_check:

Название атрибута	Тип атрибута	Смысловое значение
st_id	int 	ID ученика
sale_time	timestamp	Время покупки
money	int	Цена, по которой приобрели данный курс
subject	text 	
Необходимо в одном запросе выгрузить следующую информацию о группах пользователей:

ARPU 
ARPAU 
CR в покупку 
СR активного пользователя в покупку 
CR пользователя из активности по математике (subject = ’math’) в покупку курса по математике
ARPU считается относительно всех пользователей, попавших в группы.

Активным считается пользователь, за все время решивший больше 10 задач правильно в любых дисциплинах.

Активным по математике считается пользователь, за все время решивший 2 или больше задач правильно по математике.

Подключимся к Clich House

In [4]:
connection_default = {'host': 'https://clickhouse.lab.karpov.courses',
                      'database':'default',
                      'user':'student', 
                      'password':'dpo_python_2020'
                     }

Просмотрим на таблицу с данными

In [5]:
q = '''
    select
        st_id as st_id,
        toDateTime(timest) as timest,
        correct as correct,
        subject as subject
    from 
        peas
    limit 10    
    '''



In [6]:
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test

,st_id,timest,correct,subject
0,100379,2021-10-30 13:32:29,1,Theory of probability
1,100379,2021-10-30 14:11:19,0,Vizualization
2,100379,2021-10-30 15:54:22,1,Theory of probability
3,100379,2021-10-30 16:44:50,1,Vizualization
4,100379,2021-10-30 17:15:05,1,Theory of probability
5,100379,2021-10-30 18:02:37,1,Theory of probability
6,100379,2021-10-30 18:17:25,1,Vizualization
7,100379,2021-10-30 18:32:26,0,Theory of probability
8,100379,2021-10-30 19:19:33,1,Vizualization
9,100379,2021-10-30 19:28:03,1,Theory of probability


Проверим, за какие месяцы у нас данные 

In [7]:
q = '''
    select 
        DISTINCT YEAR(timest) as year,
        MONTH(timest) as month
    from 
        peas
    '''



In [8]:
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test

,year,month
0,2021,10


Все данные у нас за октябрь 2021г.

Теперь сделаем запрос и получим количество очень усердных студентов

In [9]:
q = '''
    select 
    count(DISTINCT st_id)
from
(
    select
        DISTINCT st_id as st_id,
        sum(correct) as correct
    from
        peas
    GROUP BY st_id
    HAVING correct>=20
    
)    

    '''

In [10]:
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test

,uniqExact(st_id)
0,136


Посмотрим данные для второй части

Дана таблицы: default.peas (см. выше), default.studs:

Название атрибута	Тип атрибута	Смысловое значение
st_id	int	 ID ученика
test_grp	text	 Метка ученика в данном эксперименте
и default.final_project_check:

Название атрибута	Тип атрибута	Смысловое значение
st_id	int 	ID ученика
sale_time	timestamp	Время покупки
money	int	Цена, по которой приобрели данный курс
subject	text 	
Необходимо в одном запросе выгрузить следующую информацию о группах пользователей:

ARPU 
ARPAU 
CR в покупку 
СR активного пользователя в покупку 
CR пользователя из активности по математике (subject = ’math’) в покупку курса по математике
ARPU считается относительно всех пользователей, попавших в группы.

Активным считается пользователь, за все время решивший больше 10 задач правильно в любых дисциплинах.

Активным по математике считается пользователь, за все время решивший 2 или больше задач правильно по математике.

In [11]:
q = '''
    select
        st_id as st_id,
        toDateTime(timest) as timest,
        correct as correct,
        subject as subject
    from 
        peas
    limit 10    
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


,st_id,timest,correct,subject
0,100379,2021-10-30 13:32:29,1,Theory of probability
1,100379,2021-10-30 14:11:19,0,Vizualization
2,100379,2021-10-30 15:54:22,1,Theory of probability
3,100379,2021-10-30 16:44:50,1,Vizualization
4,100379,2021-10-30 17:15:05,1,Theory of probability
5,100379,2021-10-30 18:02:37,1,Theory of probability
6,100379,2021-10-30 18:17:25,1,Vizualization
7,100379,2021-10-30 18:32:26,0,Theory of probability
8,100379,2021-10-30 19:19:33,1,Vizualization
9,100379,2021-10-30 19:28:03,1,Theory of probability


In [12]:
q = '''
    select
        st_id as st_id,
        toDateTime(sale_time) as sale_time,
        money as money,
        subject as subject
    from
        final_project_check 
    limit 10    
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


,st_id,sale_time,money,subject
0,101432,2021-10-31 04:44:32,85000,Math
1,101432,2021-10-31 12:43:50,65000,Vizualization
2,104885,2021-10-30 17:05:55,65000,Vizualization
3,104885,2021-10-30 22:49:33,75000,Statistics
4,106464,2021-10-31 13:17:13,85000,Math
5,114606,2021-10-31 02:26:19,75000,Statistics
6,147316,2021-10-31 11:12:27,65000,Vizualization
7,149640,2021-10-30 19:53:19,65000,Vizualization
8,230858,2021-10-31 08:14:34,85000,Math
9,269738,2021-10-30 21:33:24,70000,Theory of probability


In [13]:
q = '''
    select*
    from
        studs 
    limit 10    
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


,st_id,test_grp
0,100379,pilot
1,101432,control
2,104818,pilot
3,104885,pilot
4,104966,pilot
5,106010,pilot
6,106028,pilot
7,106464,pilot
8,106816,control
9,107250,control


Посмотрим интервал дат оплаты и все ли студенты из таблицы final_project_check оплачивали курсы

In [14]:
q = '''
    select
        min(toDateTime(sale_time)) as min_sale_time,
        max(toDateTime(sale_time)) as max_sale_time,
        max(money) as max_money,
        min(money) as min_money
    from
        final_project_check 
    limit 10    
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


,min_sale_time,max_sale_time,max_money,min_money
0,2021-10-30 15:46:33,2021-10-31 15:15:58,100000,65000


Мы видим, что все пользователи из данной таблицы оплтили курс в течении 2 дней

In [15]:
q = '''
    select
        l.st_id as st_id,
        l.toDateTime(sale_time) as sale_time,
        l.money as money,
        l.subject as subject
    from
        final_project_check 
    limit 10    
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


ClickhouseException: b"Code: 62. DB::Exception: Syntax error: failed at position 54 ('(') (line 3, col 21): (sale_time) as sale_time,\n        l.money as money,\n        l.subject as subject\n    from\n        final_project_check \n    limit 10 FORMAT TSVWithNamesAndTypes\n. Expected one of: token, Dot, UUID, DoubleColon, MOD, DIV, NOT, BETWEEN, LIKE, ILIKE, NOT LIKE, NOT ILIKE, IN, NOT IN, GLOBAL IN, GLOBAL NOT IN, IS, AND, OR, QuestionMark, alias, AS, identifier, Comma, FROM, PREWHERE, WHERE, GROUP BY, WITH, HAVING, WINDOW, ORDER BY, LIMIT, OFFSET, SETTINGS, UNION, EXCEPT, INTERSECT, INTO OUTFILE, FORMAT, end of query. (SYNTAX_ERROR) (version 21.12.2.17 (official build))\n"

In [34]:
q = '''
SELECT 
    test_grp as test_grp,
    sum(money)/count(distinct st_id) as ARPU,
    sumIf(money, correct>=10 )/countIf(distinct st_id, correct>=10) as ARPAU,
    (countIf(DISTINCT st_id, money>0)/count(DISTINCT st_id))*100 as CR,
    (countIf(DISTINCT st_id, money>0 and correct>=10)/countIf(DISTINCT st_id, correct>=10))*100 as CR_activity,
    (countIf(DISTINCT st_id, money>0 and correct>=10 and subject = 'Math')/countIf(DISTINCT st_id, correct>=10 and subject = 'Math'))*100 as CR_activity_match
FROM
    (
    SELECT 
        s.test_grp as test_grp,
        s.st_id as st_id,
        l.subject as subject,
        l.correct as correct,
        r.money as money
    from
        (
        select
            st_id as st_id,
            subject as subject,
            sum(correct) as correct
        from 
            peas
        GROUP BY
            st_id,
            subject
        ) as l
    full join
        (   
        select
            st_id as st_id,
            subject as subject,
            sum(money) as money
        from
            final_project_check
        GROUP BY
            st_id,
            subject
        ) as r
    ON
        l.st_id = r.st_id
    FULL JOIN
        (
        SELECT*
            
        From studs
        ) as s
    ON
        l.st_id = s.st_id    
    )
WHERE
    test_grp != '' and st_id !='' 
GROUP BY
    test_grp
    '''
q_test = ph.read_clickhouse(query=q, connection=connection_default)
q_test


,test_grp,ARPU,ARPAU,CR,CR_activity,CR_activity_match
0,control,7540.983607,11120.000000,4.918033,10.400000,11.764706
1,pilot,20203.389831,44175.257732,10.169492,25.773196,18.181818


Все метрики в тестовой группе выросли

In [35]:
q = '''
    SELECT* 
        --s.test_grp as test_grp,
       -- s.st_id as st_id,
       -- l.subject as subject,
      --  l.correct as correct,
       -- r.money as money
    from
        (
        select
            st_id as st_id,
            subject as subject,
            sum(correct) as correct
        from 
            peas
        GROUP BY
            st_id,
            subject
        ) as l
    full join
        (   
        select
            st_id as st_id,
            subject as subject,
            sum(money) as money
        from
            final_project_check
        GROUP BY
            st_id,
            subject
        ) as r
    ON
        l.st_id = r.st_id
    FULL JOIN
        (
        SELECT*
            
        From studs
        ) as s
    ON
        l.st_id = s.st_id
        WHERE
    test_grp != '' and st_id !='' 
        '''
    
ebanina = ph.read_clickhouse(query=q, connection=connection_default)
ebanina

ClickhouseException: b"Code: 352. DB::Exception: Ambiguous column 'st_id': While processing SELECT `--l.st_id` AS `l.st_id`, `--l.subject` AS `l.subject`, correct AS `l.correct`, `--r.st_id` AS `r.st_id`, `--r.subject` AS `r.subject`, money AS `r.money`, s.st_id AS `s.st_id`, test_grp AS `s.test_grp` FROM (SELECT st_id AS st_id, subject AS subject, sum(correct) AS correct FROM peas GROUP BY st_id, subject) AS l FULL OUTER JOIN (SELECT st_id AS st_id, subject AS subject, sum(money) AS money FROM final_project_check GROUP BY st_id, subject) AS r ON l.st_id = r.st_id FULL OUTER JOIN (SELECT * FROM studs) AS s ON l.st_id = s.st_id WHERE (test_grp != '') AND (st_id != ''). (AMBIGUOUS_COLUMN_NAME) (version 21.12.2.17 (official build))\n"

In [28]:
ebanina.money.describe()

count       530.000000
mean      15584.905660
std       31951.897898
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max      100000.000000
Name: money, dtype: float64